# C1-P1 — SAM 2.1 Hiera-L inference over the 40 frozen views

Protocol: `docs/c1_p1_multiview_proposals_protocol.md` (frozen; owner-approved).
This notebook performs ONLY the model-inference stage: 40 RGB view PNGs in,
per-view 2D masks out. Rendering, lifting, fusion, and evaluation all run
locally in the repo.

**Pins (do not edit):** sam2 @ `2b90b9f5ceec907a1c18123530e92e794ad901a4`, checkpoint
`sam2.1_hiera_large.pt` sha256 `2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318`, automatic-mask-generator
parameters exactly as in the protocol, seeds 0.

**Isolation:** upload ONLY the `views_<scene>/` folder produced by
`tools/c1p1_render.py` (RGB PNGs + manifest). Never upload semantic meshes,
`info_semantic.json`, id buffers, keys, or answers.

**Budget:** ONE run per scene, room_2 first. If room_2 fails its gates
locally, do NOT run the transfer scenes.

In [ ]:
# [1] environment + pinned checkpoint (verify sha BEFORE any inference)
import hashlib, os, subprocess, sys
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!git checkout 2b90b9f5ceec907a1c18123530e92e794ad901a4
!pip install -q -e .
!wget -q -O sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sha = hashlib.sha256(open('sam2.1_hiera_large.pt','rb').read()).hexdigest()
assert sha == '2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318', f'CHECKPOINT SHA MISMATCH: {sha}'
print('checkpoint sha OK:', sha)
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# [2] scene views: upload views_<SCENE>.tar.gz (PNGs + manifest only)
SCENE = 'replica_room_2'   # change ONLY per the staged protocol order

# M1 protocol (docs/arkitscenes_mask_coverage_protocol.md): the ONLY SAM
# parameter that may vary. 0.95 is the frozen C1-P1 pin -- leave it there
# unless a protocol says otherwise. The artifact tag keeps two
# parameterisations from ever sharing a filename.
STABILITY_SCORE_THRESH = 0.95
TAG = '' if STABILITY_SCORE_THRESH == 0.95 else \
      f".stab{int(round(STABILITY_SCORE_THRESH * 100)):03d}"
print('threshold', STABILITY_SCORE_THRESH, '| artifact tag', repr(TAG))
from google.colab import files, drive
drive.mount('/content/drive')
import tarfile, glob
tar = f'/content/drive/MyDrive/c1p1/views_{SCENE}.tar.gz'
tarfile.open(tar).extractall('/content/views')
pngs = sorted(glob.glob(f'/content/views/views_{SCENE}/view_*.png'))
assert len(pngs) == 40, f'expected 40 views, found {len(pngs)}'
print('views ready:', len(pngs))

In [ ]:
# [3] pinned automatic-mask inference (seeds 0; one pass; no retries)
# Every parameter below is frozen EXCEPT stability_score_thresh, which
# cell [2] sets. Do not edit anything else here.
import random, time, numpy as np, torch
from PIL import Image
random.seed(0); np.random.seed(0); torch.manual_seed(0)
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

model = build_sam2('configs/sam2.1/sam2.1_hiera_l.yaml',
                   'sam2.1_hiera_large.pt', device='cuda')
gen = SAM2AutomaticMaskGenerator(
    model, points_per_side=32, points_per_batch=64,
    pred_iou_thresh=0.8, stability_score_thresh=STABILITY_SCORE_THRESH,
    stability_score_offset=1.0, mask_threshold=0.0, box_nms_thresh=0.7,
    crop_n_layers=0, crop_nms_thresh=0.7, min_mask_region_area=0,
    use_m2m=False, multimask_output=True, output_mode='uncompressed_rle')

def rle_to_mask(rle):
    h, w = rle['size']
    flat = np.zeros(h * w, dtype=np.uint8)
    vals = np.zeros(len(rle['counts']), dtype=np.uint8); vals[1::2] = 1
    flat[:] = np.repeat(vals, rle['counts'])
    return flat.reshape((h, w), order='F')

out, scores, t0 = {}, {}, time.time()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    for k, p in enumerate(pngs):
        img = np.array(Image.open(p).convert('RGB'))
        anns = gen.generate(img)
        packed = np.stack([np.packbits(rle_to_mask(a['segmentation']).ravel())
                           for a in anns]) if anns else np.zeros((0, 131072), np.uint8)
        out[f'masks_{k:02d}'] = packed
        scores[f'scores_{k:02d}'] = np.array(
            [[a['predicted_iou'], a['stability_score']] for a in anns])
        print(f'view {k:02d}: {len(anns)} masks  ({time.time()-t0:.0f}s)')
elapsed = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 2**30
print(f'total {elapsed:.0f}s, peak VRAM {peak:.1f} GiB')

In [ ]:
# [4] save sidecar (packbits masks + scores + env) to Drive
import json, numpy as np, torch, platform
env = dict(scene=SCENE, sam2_commit='2b90b9f5ceec907a1c18123530e92e794ad901a4', checkpoint_sha256=sha,
           torch=torch.__version__, cuda=torch.version.cuda,
           device=torch.cuda.get_device_name(0), python=platform.python_version(),
           elapsed_seconds=round(elapsed, 1), peak_vram_gib=round(peak, 2),
           n_views=40, seeds=0,
           stability_score_thresh=STABILITY_SCORE_THRESH,
           artifact_tag=TAG)
dst = f'/content/drive/MyDrive/c1p1/c1p1_masks_{SCENE}{TAG}.npz'
np.savez_compressed(dst, env=json.dumps(env), **out, **scores)
print('saved:', dst)
print(json.dumps(env, indent=1))

## After this notebook (local, in the repo)

```
python3 tools/c1p1_fuse.py --scene replica_room_2 --masks runs/phase8_c1p1/c1p1_masks_replica_room_2.npz
python3 tools/c1p1_eval.py --scene replica_room_2
```

Gate failure on room_2 = STOP (no transfer inference).